# gold layer------------------------------

##### ==============================================================================
## 1. Gold Table: Daily Trips & Financial Summary 
##### ==============================================================================

In [ ]:
import dlt


@dlt.table(
    name="gold_daily_trips_summary",
    comment="Daily aggregated metrics for NYC taxi trips (Trip count, total fare, tips, average distance)"
)
def gold_daily_trips_summary():
    return spark.sql("""
        SELECT 
            TO_DATE(pickup_time) AS trip_date,
            COUNT(*) AS total_trips,
            ROUND(SUM(fare_amount), 2) AS total_fare_amount,
            ROUND(SUM(tips_amount), 2) AS total_tips_amount,
            ROUND(SUM(tolls_amount), 2) AS total_tolls_amount,
            ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
            ROUND(AVG(passenger_count), 2) AS avg_passengers_per_trip
        FROM LIVE.silver_trips
        GROUP BY TO_DATE(pickup_time)
        ORDER BY trip_date
    """)

##### ====================================================================
## 2. Gold Table: Zone Performance & Location Analytics 
##### ===================================================================


In [ ]:
import dlt
@dlt.table(
    name="gold_zone_performance",
    comment="Aggregated performance metrics per pickup and dropoff zones with joined zone names"
)
def gold_zone_performance():
    return spark.sql("""
        SELECT 
            t.Pickup_Location_ID,
            COALESCE(pz.Borough, 'N/A') AS pickup_borough,
            COALESCE(pz.Zone, 'N/A') AS pickup_zone,
            
            t.dropoff_Location_ID,
            COALESCE(dz.Borough, 'N/A') AS dropoff_borough,
            COALESCE(dz.Zone, 'N/A') AS dropoff_zone,
            
            COUNT(*) AS total_trips,
            ROUND(AVG(t.fare_amount), 2) AS avg_fare_amount,
            ROUND(SUM(t.tips_amount), 2) AS total_tips_amount,
            ROUND(AVG(t.trip_distance), 2) AS avg_trip_distance
            
        FROM LIVE.silver_trips t
        
        -- الـ JOIN الأول لربط بيانات منطقة الصعود (Pickup Zone)
        LEFT JOIN LIVE.silver_taxi_zones pz 
            ON t.Pickup_Location_ID = pz.location_id
            
        -- الـ JOIN الثاني لربط بيانات منطقة النزول (Dropoff Zone)
        LEFT JOIN LIVE.silver_taxi_zones dz 
            ON t.dropoff_Location_ID = dz.location_id
            
        GROUP BY 
            t.Pickup_Location_ID,
            pz.Borough,
            pz.Zone,
            t.dropoff_Location_ID,
            dz.Borough,
            dz.Zone
    """)